# 第13周 · 课堂练习：消息传递、Â 与池化的手艺活

四道小题，把课上的四个核心操作亲手写一遍。每道题都是**先在纸上算，再写代码，最后用自查对答案**——纸上算出的数和代码算出的数对得上，你才真的懂了。

| 题 | 内容 | 建议时间 |
| --- | --- | --- |
| E1 | 手写一个节点的消息传递；循环版 vs 矩阵版 | 15 分钟 |
| E2 | 构造对称归一化邻接矩阵 $\hat A$，检查它的性质 | 20 分钟 |
| E3 | 数 Cora 的统计量；量一量引用网络的同质性 | 15 分钟 |
| E4 | 按 `attention_mask` 做平均池化 | 15 分钟 |

AI 编程助手可以用（第 9 周起的规则），但这四道题的代码都只有两三行——先自己写。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：PyTorch Geometric（2.3 起是纯 Python 包，不需要 pyg-lib / torch-scatter 等编译扩展，也不要装它们）
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric>=2.8,<3"], check=True)
# transformers 在 Colab 已预装；本地环境缺它时兜底安装（课程按 5.x 的 API 编写）
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.22,<6"], check=True)
print("PyTorch Geometric 与 transformers 已就绪")

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch

warnings.filterwarnings("ignore", message=".*torch.jit.script.*")   # 新版 torch 对 PyG 内部写法的弃用提醒，与本课无关
from torch_geometric.datasets import Planetoid
from zero2researcher import data, experiments
from zero2researcher.checks import week13 as ck
from zero2researcher.checks import report

print("torch", torch.__version__)

## E1 · 手写一个节点的消息传递

下面是一张 5 个节点的小图，每个节点有 2 维特征：

In [ ]:
edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4)]
toy = nx.Graph(edges)
A_toy = torch.tensor(nx.to_numpy_array(toy, nodelist=range(5)), dtype=torch.float32)
X_toy = torch.tensor([[1., 0.], [0., 1.], [1., 1.], [2., 0.], [0., 3.]])

fig, ax = plt.subplots(figsize=(5, 2.6))
pos = {0: (0, 1), 1: (0, -1), 2: (1, 0), 3: (2, 0), 4: (3, 0)}
nx.draw_networkx(toy, pos, node_color="#9ecae1", node_size=600, ax=ax)
for v, (x, y) in pos.items():
    ax.text(x, y - 0.45, f"x = {X_toy[v].tolist()}", ha="center", fontsize=8)
ax.set_title("Toy graph: 5 nodes, 5 edges, 2 features per node")
ax.axis("off")
plt.show()
print("A_toy =\n", A_toy)

**E1a（纸上）**：节点 2 有哪几个邻居？它的邻居特征的**平均**是多少？节点 4 呢？写在下面，再往下写代码验证。

（在这里写你的手算：节点 2 的邻居 = ？，平均 = ？；节点 4 的邻居 = ？，平均 = ？）

**E1b（代码）**：写 `mean_neighbors(A, X, v)`：返回节点 `v` 的所有邻居（**不含** `v` 自己）特征的平均，形状 `(d,)`。提示：`A[v]` 是第 `v` 行；它里面等于 1 的位置就是邻居（`nonzero()`）。

In [ ]:
def mean_neighbors(A, X, v):
    # 在这里写你的代码
    raise NotImplementedError


print("节点 2 的邻居平均:", np.round(mean_neighbors(A_toy, X_toy, 2).numpy(), 4))
print("节点 4 的邻居平均:", np.round(mean_neighbors(A_toy, X_toy, 4).numpy(), 4))
ck.message_passing(mean_neighbors)

**E1c**：对所有节点循环调用 `mean_neighbors`，得到 `H_loop`（形状 (5, 2)）。再用**一次矩阵运算**写出同样的结果 `H_matrix`——平均聚合的矩阵形式是 $D^{-1}AX$：先 `A @ X`，再让每一行除以该节点的度。

In [ ]:
H_loop = torch.stack([mean_neighbors(A_toy, X_toy, v) for v in range(5)])
# 在这里写你的代码
raise NotImplementedError
print("H_matrix =\n", H_matrix)
report("循环版与矩阵版一致", torch.allclose(H_loop, H_matrix, atol=1e-6),
       "两种算法的结果不同",
       "A @ X 的第 v 行是邻居特征之和；接下来让每一行除以该节点的度——注意度向量的形状要能按行广播")

**追问**（一句话）：如果把"平均"换成"求和"，度为 168 的节点和度为 1 的节点得到的新特征，数值量级会差多少？这对下一层的训练意味着什么？

（在这里回答）

## E2 · 构造对称归一化邻接矩阵 $\hat A$

Kipf & Welling（ICLR 2017）Eq. 8：

$$\hat A = \tilde D^{-1/2}\,\tilde A\,\tilde D^{-1/2},\qquad \tilde A = A + I,\qquad \tilde D_{vv} = \sum_u \tilde A_{vu}$$

**E2a（纸上）**：对上面的小图，先写出每个节点加自环之后的度 $\tilde d_v$，再算 $\hat A_{2,3}$、$\hat A_{3,4}$ 和 $\hat A_{4,4}$（提示：$\hat A_{vu} = \tilde A_{vu} / \sqrt{\tilde d_v \tilde d_u}$）。

（在这里写：$\tilde d$ = ？；$\hat A_{2,3}$ = ？；$\hat A_{3,4}$ = ？；$\hat A_{4,4}$ = ？）

**E2b（代码）**：写 `normalized_adjacency(A)`。注意三个坑：① 不要原地修改 `A`（写 `A + torch.eye(n)`，不写 `A += ...`）；② 先算**度向量**再取 $-1/2$ 次方，再放到对角线上（`torch.diag`）——对整个度矩阵取幂会把非对角的 0 变成 inf；③ 矩阵乘法用 `@`。

In [ ]:
def normalized_adjacency(A):
    # 在这里写你的代码
    raise NotImplementedError


A_hat_toy = normalized_adjacency(A_toy)
print("Â =\n", A_hat_toy.round(decimals=4))
ck.normalized_adjacency(normalized_adjacency)

**E2c**：和你的手算对一对，然后运行下一格，看 $\hat A$ 的三个性质：

In [ ]:
print("对称:", torch.allclose(A_hat_toy, A_hat_toy.T))
print("每一行的和:", np.round(A_hat_toy.sum(dim=1).numpy(), 3))
print("特征值:", np.round(torch.linalg.eigvalsh(A_hat_toy).numpy(), 3))

A_iso = torch.zeros(6, 6)
A_iso[:5, :5] = A_toy                     # 加一个没有任何边的节点 5
print("含孤立节点时 Â 是否全是有限数:", torch.isfinite(normalized_adjacency(A_iso)).all().item())

**问题**（每题一两句）：

1. $\hat A$ 每一行的和**不等于** 1，所以它不是"加权平均"。哪一种归一化的行和恰好是 1？
2. 最大特征值是多少？如果最大特征值大于 1，把同一个矩阵乘上 8 次（8 层 GCN）会发生什么？
3. 孤立节点（度为 0）在不加自环时会让哪一步出问题？自环是怎样顺手解决它的？

（在这里回答）

## E3 · 数一数 Cora

用 PyG 加载 Cora（与讲义同一份缓存，`root=data.download_root()`），把下面五个数从 `dataset` / `cora` 里读出来（**不要手抄**）：节点数、特征维度、类别数、`edge_index` 里的边数、训练节点数。

In [ ]:
dataset = Planetoid(root=data.download_root(), name="Cora")
cora = dataset[0]
# 在这里写你的代码
raise NotImplementedError
print(f"节点 {n_nodes} | 特征 {n_features} | 类别 {n_classes} | edge_index 中的边 {n_edges} | 训练节点 {n_train}")
ck.cora_stats(n_nodes, n_features, n_classes, n_edges=n_edges, n_train=n_train)

**问题**：Kipf & Welling 的表 1 写 Cora 有 5,429 条边，你数到的是多少？差在哪里（提示：`cora.is_undirected()`；PyG 怎么存一条无向边）？

再看度分布——大多数论文只有寥寥几条引用关系：

In [ ]:
deg = torch.bincount(cora.edge_index[0], minlength=cora.num_nodes)
print(f"度：最小 {deg.min().item()}，中位数 {deg.median().item()}，最大 {deg.max().item()}；"
      f"{(deg <= 5).float().mean().item():.0%} 的论文度 ≤ 5")
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.hist(deg.numpy(), bins=np.arange(1, deg.max().item() + 2), log=True)
ax.set_xscale("log")
ax.set_xlabel("Degree (log scale)")
ax.set_ylabel("Number of papers (log scale)")
ax.set_title("Heavy tail: most papers have at most 5 links, one has 168")
plt.tight_layout()
plt.show()

**E3b · 同质性**：GCN 之所以有用，是因为"相连的节点往往同类"。量一量：在 `edge_index` 的所有边里，**两端节点主题相同**的比例 `edge_homophily` 是多少？（`src, dst = cora.edge_index`，比较 `cora.y[src]` 与 `cora.y[dst]`。）作为对照，下面也算出了"随机连边"时两端同类的概率 $\sum_c p_c^2$。

In [ ]:
src, dst = cora.edge_index
# 在这里写你的代码
raise NotImplementedError
p = torch.bincount(cora.y).float() / cora.num_nodes          # 每个主题的占比
random_baseline = (p ** 2).sum().item()
print(f"同类边的比例 {edge_homophily:.3f} | 如果随机连边，期望只有 {random_baseline:.3f}")
report("Cora 的边同质性", 0.78 <= edge_homophily <= 0.84,
       f"期望约 0.81，实际 {edge_homophily:.3f}",
       "对每一条边比较两端节点的标签是否相同，再求\"相同\"所占的比例（布尔张量要先转成浮点数才能求平均）")

**追问**（一两句）：如果一张图的同质性只有 0.2（相连的节点大多**不**同类，例如"异性相吸"的约会网络），你预期 GCN 相对 MLP 还会有优势吗？为什么？

（在这里回答）

## E4 · 按 attention_mask 做平均池化

一个 batch 里的句子长短不一，分词器用 `[PAD]` 把它们补到一样长，并给出 `attention_mask`（真实 token 为 1，padding 为 0）。Transformer 的输出 `last_hidden_state` 形状 (B, T, H)，**padding 位置也有输出向量**（而且不是 0）——平均池化时必须把它们排除。

下面是一个手造的小例子：2 个句子、长度补到 4、每个 token 2 维；padding 位置故意填了 100 当"垃圾"。

**E4a（纸上）**：两个句子按 mask 平均之后分别是多少？如果不管 mask 直接平均呢？

In [ ]:
h = torch.tensor([[[1., 2.], [3., 4.], [5., 6.], [100., 100.]],
                  [[2., 0.], [4., 0.], [100., 100.], [100., 100.]]])     # (B=2, T=4, H=2)
mask = torch.tensor([[1, 1, 1, 0],
                     [1, 1, 0, 0]])                                      # (B=2, T=4)
print("h 的形状:", tuple(h.shape), "| mask 的形状:", tuple(mask.shape))
print("不管 mask 直接平均:\n", h.mean(dim=1))

（在这里写你的手算）

**E4b（代码）**：写 `mean_pool(last_hidden_state, attention_mask)` → (B, H)。思路：把 mask 变成 (B, T, 1) 的浮点张量 `m`，`(h * m).sum(dim=1)` 是真实 token 之和，再除以每句的真实 token 数 `m.sum(dim=1)`。

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    # 在这里写你的代码
    raise NotImplementedError


print("按 mask 平均:\n", mean_pool(h, mask))
ck.mean_pool(mean_pool)

**问题**（一两句）：① 不管 mask 的平均，对短句子还是长句子伤害更大？② `[CLS]` 池化（直接取 `h[:, 0]`）需要 mask 吗？为什么？

**复盘**：本周四个操作——邻居平均、$\hat A$、按 mask 平均、`[CLS]` 取第一个——其实都是"对一组向量做加权求和"。分别写出它们的权重是什么。

（在这里回答）